# RAG Pipeline for Docs (Zero Hallucination)

This notebook implements the full RAG pipeline as described in the architecture diagram, covering all 10 blocks:

1. **Ingest + Normalize Docs** — Deduplication, format standardization, metadata extraction, versioning
2. **Hybrid Retrieval (BM25 + Embeddings)** — Keyword + semantic search indexes
3. **ANN Retrieval + Reranking** — Fast candidate fetch + cross-encoder reranking
4. **Source Confidence Scoring** — Freshness, trust, overlap, retrieval consistency
5. **Constrained Generation** — LLM answers ONLY from retrieved context
6. **Citation-Backed Responses** — Every claim linked to source chunks
7. **Confidence Threshold Gate** — Refuse to answer if evidence is insufficient
8. **Continuous Evals** — Adversarial queries, hallucination rate tracking
9. **Caching + Memory Layer** — Cache frequent queries and retrieval results
10. **Observability** — Trace retrieval path, chunk rankings, token attribution


## Setup & Installations

In [1]:
# Install required packages
!pip install -q \
    rank_bm25 \
    sentence-transformers \
    faiss-cpu \
    anthropic \
    numpy \
    pandas \
    tqdm \
    hashlib \
    diskcache \
    rich

print("All packages installed.")

All packages installed.


ERROR: Ignored the following yanked versions: 20081119
ERROR: Could not find a version that satisfies the requirement hashlib (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for hashlib


In [2]:
# Core imports
import os
import re
import json
import time
import hashlib
import logging
import datetime
from copy import deepcopy
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any

import numpy as np
import pandas as pd
from tqdm import tqdm
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

# BM25
from rank_bm25 import BM25Okapi

# Embeddings & ANN
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

# Caching
import diskcache as dc

# Anthropic
import anthropic

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)
console = Console()

# ── Anthropic client ──────────────────────────────────────────────────────────
# Set your key: export ANTHROPIC_API_KEY=sk-...
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "YOUR_API_KEY_HERE")
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

console.print("[bold green]✓ Imports complete[/bold green]")

ModuleNotFoundError: No module named 'rank_bm25'

---
## Block 1 — Ingest + Normalize Docs
> Deduplication · Format standardization · Metadata extraction · Versioning

In [ ]:
@dataclass
class Document:
    """Normalized document unit with full metadata."""
    doc_id: str
    content: str
    source: str
    title: str = ""
    page: Optional[int] = None
    chunk_index: int = 0
    created_at: str = field(default_factory=lambda: datetime.datetime.utcnow().isoformat())
    updated_at: str = field(default_factory=lambda: datetime.datetime.utcnow().isoformat())
    version: int = 1
    trust_score: float = 1.0       # domain-level trustworthiness [0,1]
    content_hash: str = ""
    tokens: List[str] = field(default_factory=list)

    def __post_init__(self):
        self.content_hash = hashlib.sha256(self.content.encode()).hexdigest()
        self.tokens = self._tokenize(self.content)

    @staticmethod
    def _tokenize(text: str) -> List[str]:
        return re.sub(r"[^a-z0-9\s]", "", text.lower()).split()


class DocIngester:
    """Block 1: Ingest, deduplicate, normalize, version documents."""

    def __init__(self, chunk_size: int = 300, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self._seen_hashes: set = set()
        self._version_map: Dict[str, int] = {}

    # ── public API ────────────────────────────────────────────────────────────
    def ingest(self, raw_docs: List[Dict]) -> List[Document]:
        """Ingest a list of raw doc dicts and return normalized, chunked Documents."""
        all_chunks: List[Document] = []
        for raw in tqdm(raw_docs, desc="Ingesting docs"):
            normalized = self._normalize(raw)
            chunks = self._chunk(normalized)
            deduped = self._deduplicate(chunks)
            versioned = self._version(deduped)
            all_chunks.extend(versioned)
        logger.info(f"Ingested {len(all_chunks)} chunks from {len(raw_docs)} documents.")
        return all_chunks

    # ── private helpers ───────────────────────────────────────────────────────
    def _normalize(self, raw: Dict) -> Dict:
        """Standardize format: strip HTML tags, normalize whitespace."""
        text = raw.get("content", "")
        text = re.sub(r"<[^>]+>", " ", text)          # strip HTML
        text = re.sub(r"\s+", " ", text).strip()        # normalize whitespace
        raw["content"] = text
        # Extract metadata
        raw.setdefault("source", "unknown")
        raw.setdefault("title", raw["source"])
        raw.setdefault("trust_score", 1.0)
        return raw

    def _chunk(self, raw: Dict) -> List[Document]:
        """Split long text into overlapping chunks."""
        words = raw["content"].split()
        chunks = []
        start = 0
        idx = 0
        while start < len(words):
            end = min(start + self.chunk_size, len(words))
            chunk_text = " ".join(words[start:end])
            doc_id = f"{raw['source']}::chunk_{idx}"
            chunks.append(Document(
                doc_id=doc_id,
                content=chunk_text,
                source=raw["source"],
                title=raw["title"],
                page=raw.get("page"),
                chunk_index=idx,
                trust_score=raw["trust_score"],
            ))
            if end == len(words):
                break
            start += self.chunk_size - self.chunk_overlap
            idx += 1
        return chunks

    def _deduplicate(self, chunks: List[Document]) -> List[Document]:
        """Drop exact-duplicate chunks based on content hash."""
        unique = []
        for c in chunks:
            if c.content_hash not in self._seen_hashes:
                self._seen_hashes.add(c.content_hash)
                unique.append(c)
        return unique

    def _version(self, chunks: List[Document]) -> List[Document]:
        """Assign/increment version numbers per source."""
        for c in chunks:
            key = c.source
            self._version_map[key] = self._version_map.get(key, 0) + 1
            c.version = self._version_map[key]
        return chunks


# ── Demo data ─────────────────────────────────────────────────────────────────
RAW_DOCS = [
    {
        "source": "Doc A", "page": 12, "trust_score": 0.95,
        "title": "Introduction to RAG",
        "content": (
            "Retrieval-Augmented Generation (RAG) is a framework that combines information retrieval "
            "with large language models. Instead of relying solely on parametric knowledge baked into "
            "model weights during training, RAG systems first retrieve relevant documents from an "
            "external corpus and then condition the language model on those retrieved passages. "
            "This approach significantly reduces hallucination because the model can ground its "
            "answers in verifiable external evidence. RAG was first introduced by Lewis et al. in 2020 "
            "and has since become the dominant architecture for knowledge-intensive NLP tasks."
        )
    },
    {
        "source": "Doc B", "page": 7, "trust_score": 0.90,
        "title": "Vector Search and Embeddings",
        "content": (
            "Dense vector search, also called semantic search or neural search, relies on embedding "
            "models to represent text as high-dimensional numerical vectors. The cosine similarity or "
            "inner product between two vectors reflects their semantic proximity. Modern embedding "
            "models such as text-embedding-ada-002 from OpenAI or the open-source BGE family from "
            "BAAI produce vectors of 768 to 1536 dimensions. Approximate Nearest Neighbor (ANN) "
            "indexes such as FAISS, ScaNN, and Hnswlib enable sub-millisecond retrieval even over "
            "billions of vectors by trading a small amount of recall for large gains in throughput."
        )
    },
    {
        "source": "Doc C", "page": 23, "trust_score": 0.85,
        "title": "BM25 and Hybrid Retrieval",
        "content": (
            "BM25 (Best Match 25) is a probabilistic keyword-based retrieval function that ranks "
            "documents by their relevance to a query using term frequency and inverse document "
            "frequency statistics. It is an improvement over TF-IDF and remains the strongest "
            "keyword baseline for information retrieval benchmarks. Hybrid retrieval combines BM25 "
            "scores with dense embedding scores, typically through reciprocal rank fusion (RRF) or "
            "a learned weighted combination. Empirical studies consistently show that hybrid "
            "retrieval outperforms either method alone, particularly on queries containing rare "
            "technical terms or proper nouns that embeddings may not capture well."
        )
    },
    {
        "source": "Doc A", "page": 15, "trust_score": 0.95,  # same source, different page
        "title": "Introduction to RAG — Evaluation",
        "content": (
            "Evaluating RAG systems requires measuring both retrieval quality and generation quality. "
            "Retrieval metrics include Recall@K, NDCG, and MRR. Generation metrics include faithfulness "
            "(does the answer follow from the retrieved context?), relevance, and completeness. "
            "Frameworks like RAGAS, TruLens, and DeepEval automate the evaluation pipeline. "
            "A faithfulness score below 0.8 typically indicates hallucination risk. "
            "Continuous evaluation with adversarial queries is essential for production RAG systems "
            "to catch regressions early."
        )
    },
    {
        "source": "Doc D", "page": 3, "trust_score": 0.80,
        "title": "Reranking with Cross-Encoders",
        "content": (
            "Cross-encoder reranking is a two-stage retrieval strategy. In stage one, a fast bi-encoder "
            "retrieves hundreds of candidate passages. In stage two, a cross-encoder model such as "
            "ms-marco-MiniLM-L-6-v2 or BGE-Reranker scores each query-passage pair jointly, allowing "
            "full attention between the query and passage tokens. Cross-encoders are significantly more "
            "accurate than bi-encoders but much slower, making the two-stage design necessary. "
            "Typical production setups retrieve Top-100 with ANN and rerank to Top-5 or Top-10."
        )
    },
]

ingester = DocIngester(chunk_size=120, chunk_overlap=20)
all_docs: List[Document] = ingester.ingest(RAW_DOCS)

console.print(Panel(
    f"[bold]Ingested [cyan]{len(all_docs)}[/cyan] chunks[/bold]\n"
    + "\n".join(f"  • [{d.source} p.{d.page}] chunk {d.chunk_index} — v{d.version} — {len(d.tokens)} tokens"
               for d in all_docs),
    title="[1] Ingest + Normalize Docs", border_style="purple"
))

---
## Block 2 — Hybrid Retrieval (BM25 + Embeddings)
> Keyword matching with BM25 · Semantic search with dense embeddings

In [ ]:
class HybridRetriever:
    """
    Block 2: BM25 index (keyword) + FAISS vector index (semantic).
    Returns fused candidate chunks via Reciprocal Rank Fusion (RRF).
    """

    def __init__(self, embed_model_name: str = "all-MiniLM-L6-v2", rrf_k: int = 60):
        logger.info(f"Loading embedding model: {embed_model_name}")
        self.embed_model = SentenceTransformer(embed_model_name)
        self.rrf_k = rrf_k
        self.docs: List[Document] = []
        self.bm25: Optional[BM25Okapi] = None
        self.faiss_index: Optional[faiss.IndexFlatIP] = None
        self.embeddings: Optional[np.ndarray] = None

    # ── Build indexes ─────────────────────────────────────────────────────────
    def build(self, docs: List[Document]) -> None:
        self.docs = docs
        corpus_tokens = [d.tokens for d in docs]
        self.bm25 = BM25Okapi(corpus_tokens)
        logger.info("BM25 index built.")

        texts = [d.content for d in docs]
        self.embeddings = self.embed_model.encode(
            texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True
        ).astype("float32")
        dim = self.embeddings.shape[1]
        self.faiss_index = faiss.IndexFlatIP(dim)  # inner product == cosine for normalized vecs
        self.faiss_index.add(self.embeddings)
        logger.info(f"FAISS index built with {self.faiss_index.ntotal} vectors (dim={dim}).")

    # ── Retrieve ──────────────────────────────────────────────────────────────
    def retrieve(self, query: str, top_k: int = 10) -> List[Tuple[Document, float]]:
        """Return top_k docs sorted by RRF-fused score."""
        bm25_ranked = self._bm25_rank(query, top_k * 2)
        dense_ranked = self._dense_rank(query, top_k * 2)
        fused = self._rrf(bm25_ranked, dense_ranked)
        return fused[:top_k]

    # ── Private helpers ───────────────────────────────────────────────────────
    def _bm25_rank(self, query: str, k: int) -> List[Tuple[int, float]]:
        tokens = Document._tokenize(query)
        scores = self.bm25.get_scores(tokens)
        ranked_idx = np.argsort(scores)[::-1][:k]
        return [(int(i), float(scores[i])) for i in ranked_idx]

    def _dense_rank(self, query: str, k: int) -> List[Tuple[int, float]]:
        q_vec = self.embed_model.encode([query], normalize_embeddings=True).astype("float32")
        sims, indices = self.faiss_index.search(q_vec, k)
        return [(int(indices[0][i]), float(sims[0][i])) for i in range(len(indices[0]))]

    def _rrf(self,
             bm25_ranked: List[Tuple[int, float]],
             dense_ranked: List[Tuple[int, float]]) -> List[Tuple[Document, float]]:
        scores: Dict[int, float] = {}
        for rank, (idx, _) in enumerate(bm25_ranked):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (self.rrf_k + rank + 1)
        for rank, (idx, _) in enumerate(dense_ranked):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (self.rrf_k + rank + 1)
        sorted_idx = sorted(scores, key=scores.__getitem__, reverse=True)
        return [(self.docs[i], scores[i]) for i in sorted_idx]


retriever = HybridRetriever()
retriever.build(all_docs)

# Quick sanity check
QUERY = "What is BM25 and how does it combine with dense embeddings?"
hybrid_results = retriever.retrieve(QUERY, top_k=6)

t = Table(title="[2] Hybrid Retrieval Results", show_lines=True)
t.add_column("Rank"); t.add_column("Source"); t.add_column("RRF Score"); t.add_column("Snippet")
for rank, (doc, score) in enumerate(hybrid_results, 1):
    t.add_row(str(rank), f"{doc.source} p.{doc.page}", f"{score:.4f}", doc.content[:80] + "…")
console.print(t)

---
## Block 3 — ANN Retrieval + Reranking
> ANN quickly fetches Top-K candidates · Cross-encoder reranker filters & reorders by relevance

In [ ]:
class Reranker:
    """
    Block 3: Cross-encoder reranker (e.g. MS-MARCO MiniLM or BGE reranker).
    Takes Top-K from hybrid retrieval, returns Top-N reranked by relevance.
    """

    def __init__(self, model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
        logger.info(f"Loading cross-encoder: {model_name}")
        self.model = CrossEncoder(model_name)

    def rerank(self,
               query: str,
               candidates: List[Tuple[Document, float]],
               top_n: int = 4) -> List[Tuple[Document, float]]:
        """Score each (query, passage) pair and return top_n."""
        if not candidates:
            return []
        pairs = [(query, doc.content) for doc, _ in candidates]
        scores = self.model.predict(pairs)
        ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
        return [(doc, float(score)) for (doc, _), score in ranked[:top_n]]


reranker = Reranker()
reranked_results = reranker.rerank(QUERY, hybrid_results, top_n=4)

t = Table(title="[3] After Cross-Encoder Reranking", show_lines=True)
t.add_column("Rank"); t.add_column("Source"); t.add_column("CE Score"); t.add_column("Snippet")
for rank, (doc, score) in enumerate(reranked_results, 1):
    t.add_row(str(rank), f"{doc.source} p.{doc.page}", f"{score:.4f}", doc.content[:80] + "…")
console.print(t)

---
## Block 4 — Source Confidence Scoring
> Score by freshness, trust, overlap, retrieval consistency · Filter low-confidence chunks

In [ ]:
@dataclass
class ScoredChunk:
    doc: Document
    rerank_score: float
    freshness_score: float
    trust_score: float
    overlap_score: float
    consistency_score: float
    final_confidence: float


class ConfidenceScorer:
    """
    Block 4: Combines multiple signals into a per-chunk confidence score.
      - freshness:    newer docs score higher
      - trust:        source-level authority (set at ingest time)
      - overlap:      lexical overlap between query and chunk
      - consistency:  reranker score normalized
    """

    def __init__(self,
                 freshness_half_life_days: float = 365,
                 min_confidence: float = 0.30,
                 weights: Dict[str, float] = None):
        self.half_life = freshness_half_life_days
        self.min_confidence = min_confidence
        self.weights = weights or {
            "freshness": 0.15,
            "trust": 0.25,
            "overlap": 0.25,
            "consistency": 0.35,
        }

    def score(self,
              query: str,
              chunks: List[Tuple[Document, float]]) -> List[ScoredChunk]:
        """Return ScoredChunk list, filtered to those above min_confidence."""
        query_tokens = set(Document._tokenize(query))

        # Normalise reranker scores to [0,1]
        raw_scores = np.array([s for _, s in chunks])
        if raw_scores.max() != raw_scores.min():
            norm_scores = (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())
        else:
            norm_scores = np.ones(len(raw_scores))

        scored = []
        for (doc, raw_s), norm_s in zip(chunks, norm_scores):
            freshness = self._freshness(doc)
            trust = doc.trust_score
            overlap = self._overlap(query_tokens, doc)
            consistency = float(norm_s)

            final = (
                self.weights["freshness"]    * freshness +
                self.weights["trust"]        * trust +
                self.weights["overlap"]      * overlap +
                self.weights["consistency"]  * consistency
            )
            scored.append(ScoredChunk(
                doc=doc,
                rerank_score=raw_s,
                freshness_score=freshness,
                trust_score=trust,
                overlap_score=overlap,
                consistency_score=consistency,
                final_confidence=final,
            ))

        # Filter low-confidence chunks
        filtered = [sc for sc in scored if sc.final_confidence >= self.min_confidence]
        filtered.sort(key=lambda x: x.final_confidence, reverse=True)
        logger.info(
            f"Confidence scoring: {len(chunks)} in → {len(filtered)} passed "
            f"(threshold={self.min_confidence})"
        )
        return filtered

    # ── helpers ───────────────────────────────────────────────────────────────
    def _freshness(self, doc: Document) -> float:
        """Exponential decay from updated_at date."""
        try:
            updated = datetime.datetime.fromisoformat(doc.updated_at)
            age_days = (datetime.datetime.utcnow() - updated).days
            return float(np.exp(-np.log(2) * age_days / self.half_life))
        except Exception:
            return 0.5

    @staticmethod
    def _overlap(query_tokens: set, doc: Document) -> float:
        doc_tokens = set(doc.tokens)
        if not doc_tokens:
            return 0.0
        return len(query_tokens & doc_tokens) / len(query_tokens | doc_tokens)


conf_scorer = ConfidenceScorer(min_confidence=0.25)
scored_chunks: List[ScoredChunk] = conf_scorer.score(QUERY, reranked_results)

t = Table(title="[4] Source Confidence Scores", show_lines=True)
for col in ["Source", "Fresh", "Trust", "Overlap", "Consist.", "Final Conf."]:
    t.add_column(col)
for sc in scored_chunks:
    t.add_row(
        f"{sc.doc.source} p.{sc.doc.page}",
        f"{sc.freshness_score:.2f}",
        f"{sc.trust_score:.2f}",
        f"{sc.overlap_score:.2f}",
        f"{sc.consistency_score:.2f}",
        f"[bold cyan]{sc.final_confidence:.3f}[/bold cyan]",
    )
console.print(t)

---
## Block 7 — Confidence Threshold Gate
> If confidence < threshold → return "Insufficient Evidence" (no answer)

In [ ]:
class ConfidenceGate:
    """
    Block 7: Hard gate — if the best chunk confidence is below threshold,
    refuse to generate and return InsufficientEvidence.
    """

    INSUFFICIENT = "__INSUFFICIENT_EVIDENCE__"

    def __init__(self, threshold: float = 0.30):
        self.threshold = threshold

    def check(self, scored: List[ScoredChunk]) -> Tuple[bool, str]:
        """
        Returns (passed, reason).
        passed=False means we should NOT call the LLM.
        """
        if not scored:
            return False, "No chunks retrieved."
        best = scored[0].final_confidence
        if best < self.threshold:
            return False, (
                f"Best chunk confidence {best:.3f} < threshold {self.threshold}. "
                "Insufficient evidence to answer reliably."
            )
        return True, f"Best confidence {best:.3f} ≥ threshold {self.threshold}. Proceeding."


gate = ConfidenceGate(threshold=0.30)
gate_passed, gate_reason = gate.check(scored_chunks)

color = "green" if gate_passed else "bold red"
icon  = "✓" if gate_passed else "✗  INSUFFICIENT EVIDENCE FOUND (NO ANSWER)"
console.print(Panel(f"[{color}]{icon}[/{color}]\n{gate_reason}",
                    title="[7] Confidence Gate", border_style="red" if not gate_passed else "green"))

---
## Block 5 — Constrained Generation
> LLM answers ONLY from retrieved context · No external knowledge or assumptions

In [ ]:
SYSTEM_PROMPT = """\
You are a precise document-answering assistant that operates with ZERO HALLUCINATION.

STRICT RULES:
1. Answer ONLY using information explicitly present in the provided context chunks.
2. NEVER use any external knowledge, assumptions, or information from your training data.
3. For EVERY factual claim you make, you MUST cite the source chunk using the format [Source Name, p.PAGE, chunk N].
4. If the provided context does not contain sufficient information to answer the question, you MUST respond with:
   "I cannot answer this question based on the provided documents."
5. Do NOT speculate, infer, or extrapolate beyond what is explicitly stated in the context.
6. Structure your answer with:
   - A direct answer to the question
   - Supporting evidence with citations
   - A CITATIONS section at the end listing all sources used
"""


class ConstrainedGenerator:
    """
    Block 5: LLM generation strictly constrained to retrieved context.
    Uses Claude with a strict system prompt to prevent hallucination.
    """

    def __init__(self, model: str = "claude-sonnet-4-6", max_tokens: int = 1024):
        self.model = model
        self.max_tokens = max_tokens

    def generate(self,
                 query: str,
                 scored_chunks: List[ScoredChunk]) -> Dict[str, Any]:
        """Generate a grounded answer. Returns dict with answer + raw response."""
        if not scored_chunks:
            return {"answer": gate.INSUFFICIENT, "chunks_used": [], "raw": None}

        context_block = self._build_context(scored_chunks)
        user_message = f"""CONTEXT DOCUMENTS:
---
{context_block}
---

QUESTION: {query}

Remember: cite every claim with [Source, p.PAGE, chunk N]. If context is insufficient, say so."""

        response = client.messages.create(
            model=self.model,
            max_tokens=self.max_tokens,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_message}],
        )
        answer = response.content[0].text
        return {
            "answer": answer,
            "chunks_used": scored_chunks,
            "raw": response,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
        }

    @staticmethod
    def _build_context(scored_chunks: List[ScoredChunk]) -> str:
        parts = []
        for i, sc in enumerate(scored_chunks):
            d = sc.doc
            parts.append(
                f"[Chunk {i+1}] Source: {d.source}, Page: {d.page}, "
                f"Chunk Index: {d.chunk_index}, Confidence: {sc.final_confidence:.3f}\n"
                f"Title: {d.title}\n"
                f"Content: {d.content}"
            )
        return "\n\n".join(parts)


generator = ConstrainedGenerator()

if gate_passed:
    gen_result = generator.generate(QUERY, scored_chunks)
    console.print(Panel(
        gen_result["answer"],
        title="[5] Constrained Generation — Raw Answer",
        border_style="blue"
    ))
    console.print(f"Tokens used — input: {gen_result['input_tokens']}, output: {gen_result['output_tokens']}")
else:
    gen_result = {"answer": ConfidenceGate.INSUFFICIENT, "chunks_used": []}
    console.print("[bold red]Generation skipped — gate did not pass.[/bold red]")

---
## Block 6 — Citation-Backed Responses
> Every claim linked to source chunks, docs, timestamps · Outputs are auditable

In [ ]:
@dataclass
class CitationRecord:
    index: int
    source: str
    page: Optional[int]
    chunk_index: int
    title: str
    confidence: float
    content_snippet: str
    retrieved_at: str = field(default_factory=lambda: datetime.datetime.utcnow().isoformat())


class CitationBuilder:
    """
    Block 6: Build an auditable citation list from the scored chunks
    and attach them to the final response.
    """

    def build(self, scored_chunks: List[ScoredChunk]) -> List[CitationRecord]:
        records = []
        seen = set()
        for i, sc in enumerate(scored_chunks):
            key = (sc.doc.source, sc.doc.page, sc.doc.chunk_index)
            if key in seen:
                continue
            seen.add(key)
            records.append(CitationRecord(
                index=i + 1,
                source=sc.doc.source,
                page=sc.doc.page,
                chunk_index=sc.doc.chunk_index,
                title=sc.doc.title,
                confidence=sc.final_confidence,
                content_snippet=sc.doc.content[:120] + "…",
            ))
        return records

    def format_citation_footer(self, records: List[CitationRecord]) -> str:
        lines = ["\n--- CITATIONS ---"]
        for r in records:
            lines.append(
                f"[{r.index}] {r.source} (p. {r.page}, chunk {r.chunk_index}) — "
                f"'{r.title}' | confidence: {r.confidence:.3f} | retrieved: {r.retrieved_at}"
            )
        return "\n".join(lines)


citation_builder = CitationBuilder()
citations = citation_builder.build(gen_result.get("chunks_used", []))
citation_footer = citation_builder.format_citation_footer(citations)

# Final response = answer + citations
FINAL_RESPONSE = gen_result["answer"] + "\n" + citation_footer

console.print(Panel(FINAL_RESPONSE, title="[6] Citation-Backed Final Response", border_style="green"))

---
## Block 8 — Continuous Evals
> Adversarial queries · Retrieval recall benchmarks · Hallucination rate tracking · Offline + online evaluation

In [ ]:
class EvalSuite:
    """
    Block 8: Automated evaluation suite.
    Tracks faithfulness, retrieval recall, and hallucination rate.
    """

    def __init__(self):
        self.results: List[Dict] = []

    # ── Faithfulness check ────────────────────────────────────────────────────
    def faithfulness_check(self, answer: str, chunks: List[ScoredChunk]) -> float:
        """
        Heuristic faithfulness: what fraction of answer sentences
        contain at least one token from the retrieved chunks?
        """
        if not answer or chunks is None:
            return 0.0
        context_tokens = set()
        for sc in chunks:
            context_tokens.update(sc.doc.tokens)

        sentences = [s.strip() for s in re.split(r'[.!?]', answer) if s.strip()]
        if not sentences:
            return 1.0
        grounded = sum(
            1 for s in sentences
            if any(tok in context_tokens for tok in Document._tokenize(s))
        )
        return grounded / len(sentences)

    # ── Retrieval recall ──────────────────────────────────────────────────────
    def retrieval_recall_at_k(
        self,
        retrieved: List[Document],
        relevant_sources: List[str],
        k: int = 4
    ) -> float:
        """Fraction of known-relevant sources found in top-k retrieved docs."""
        top_k_sources = {d.source for d in retrieved[:k]}
        found = sum(1 for s in relevant_sources if s in top_k_sources)
        return found / len(relevant_sources) if relevant_sources else 0.0

    # ── Run a single evaluation case ──────────────────────────────────────────
    def run_case(self,
                 query: str,
                 answer: str,
                 chunks: List[ScoredChunk],
                 known_relevant: List[str],
                 gate_passed: bool) -> Dict:
        faith = self.faithfulness_check(answer, chunks)
        recall = self.retrieval_recall_at_k([sc.doc for sc in chunks], known_relevant)
        hallucination_risk = 1.0 - faith
        result = {
            "query": query,
            "gate_passed": gate_passed,
            "faithfulness": faith,
            "retrieval_recall": recall,
            "hallucination_risk": hallucination_risk,
            "chunks_used": len(chunks),
            "timestamp": datetime.datetime.utcnow().isoformat(),
        }
        self.results.append(result)
        return result

    # ── Adversarial tests ─────────────────────────────────────────────────────
    def adversarial_tests(self,
                          retriever: HybridRetriever,
                          reranker: Reranker,
                          scorer: ConfidenceScorer,
                          gate: ConfidenceGate) -> pd.DataFrame:
        """Run a suite of adversarial queries designed to trigger hallucination."""
        tests = [
            # (query, should_pass, known_relevant_sources)
            ("What is BM25 and how does it combine with embeddings?",
             True,  ["Doc C", "Doc B"]),
            ("Who invented the internet?",                           # out-of-corpus
             False, []),
            ("What is the capital of France?",                       # out-of-corpus
             False, []),
            ("What is retrieval-augmented generation?",
             True,  ["Doc A"]),
            ("Explain quantum entanglement",                         # out-of-corpus
             False, []),
        ]

        rows = []
        for query, expected_pass, known_relevant in tests:
            hy = retriever.retrieve(query, top_k=6)
            rr = reranker.rerank(query, hy, top_n=4)
            sc = scorer.score(query, rr)
            passed, reason = gate.check(sc)
            rows.append({
                "Query": query[:55],
                "Expected Pass": expected_pass,
                "Actual Pass": passed,
                "Correct Gate": passed == expected_pass,
                "Best Confidence": f"{sc[0].final_confidence:.3f}" if sc else "N/A",
                "Chunks Retrieved": len(sc),
            })
        df = pd.DataFrame(rows)
        accuracy = df["Correct Gate"].mean() * 100
        logger.info(f"Adversarial gate accuracy: {accuracy:.0f}%")
        return df

    def summary(self) -> pd.DataFrame:
        return pd.DataFrame(self.results)


eval_suite = EvalSuite()

# Run eval on our main query
eval_result = eval_suite.run_case(
    query=QUERY,
    answer=gen_result["answer"],
    chunks=scored_chunks,
    known_relevant=["Doc B", "Doc C"],
    gate_passed=gate_passed,
)

console.print(Panel(
    f"Faithfulness:        [cyan]{eval_result['faithfulness']:.2%}[/cyan]\n"
    f"Retrieval Recall@4: [cyan]{eval_result['retrieval_recall']:.2%}[/cyan]\n"
    f"Hallucination Risk: [{'red' if eval_result['hallucination_risk']>0.2 else 'green'}]"
    f"{eval_result['hallucination_risk']:.2%}[/]",
    title="[8] Continuous Evals — Main Query", border_style="yellow"
))

# Adversarial suite
adv_df = eval_suite.adversarial_tests(retriever, reranker, conf_scorer, gate)
console.print("\n[bold yellow]Adversarial Test Results:[/bold yellow]")
print(adv_df.to_string(index=False))

---
## Block 9 — Caching + Memory Layer
> Cache frequent queries · Cache retrieval results · User/session memory for personalization

In [ ]:
class CacheMemoryLayer:
    """
    Block 9: Two-level cache:
      L1 — in-memory dict (fastest, session-scoped)
      L2 — disk cache (persistent across sessions, via diskcache)
    Also maintains per-session memory for personalization.
    """

    def __init__(self, cache_dir: str = "/tmp/rag_cache", ttl_seconds: int = 3600):
        self._l1: Dict[str, Any] = {}              # in-memory
        self._l2 = dc.Cache(cache_dir)             # disk-based
        self.ttl = ttl_seconds
        self._session_memory: Dict[str, List[Dict]] = {}  # user_id → history
        self.stats = {"l1_hits": 0, "l2_hits": 0, "misses": 0}

    # ── Query cache ───────────────────────────────────────────────────────────
    def get(self, query: str) -> Optional[Dict]:
        key = self._hash(query)
        if key in self._l1:
            self.stats["l1_hits"] += 1
            return self._l1[key]
        val = self._l2.get(key)
        if val is not None:
            self._l1[key] = val        # promote to L1
            self.stats["l2_hits"] += 1
            return val
        self.stats["misses"] += 1
        return None

    def set(self, query: str, result: Dict) -> None:
        key = self._hash(query)
        # Store only serialisable parts
        payload = {
            "answer": result.get("answer"),
            "citations": [(c.source, c.page, c.chunk_index) for c in result.get("citations", [])],
            "cached_at": datetime.datetime.utcnow().isoformat(),
        }
        self._l1[key] = payload
        self._l2.set(key, payload, expire=self.ttl)

    # ── Session memory ────────────────────────────────────────────────────────
    def add_to_session(self, user_id: str, query: str, answer: str) -> None:
        if user_id not in self._session_memory:
            self._session_memory[user_id] = []
        self._session_memory[user_id].append({
            "query": query,
            "answer": answer[:200],
            "ts": datetime.datetime.utcnow().isoformat(),
        })

    def get_session_context(self, user_id: str, last_n: int = 3) -> List[Dict]:
        return self._session_memory.get(user_id, [])[-last_n:]

    def cache_stats(self) -> Dict:
        total = sum(self.stats.values())
        hit_rate = (self.stats["l1_hits"] + self.stats["l2_hits"]) / max(total, 1)
        return {**self.stats, "total": total, "hit_rate": f"{hit_rate:.1%}"}

    @staticmethod
    def _hash(text: str) -> str:
        return hashlib.md5(text.strip().lower().encode()).hexdigest()


cache = CacheMemoryLayer()

# Simulate cache miss (first run), then cache hit (second run)
cached = cache.get(QUERY)
console.print(f"Cache lookup 1: {'HIT' if cached else 'MISS'}")

cache.set(QUERY, {"answer": gen_result["answer"], "citations": citations})
cache.add_to_session("user_001", QUERY, gen_result["answer"])

cached2 = cache.get(QUERY)
console.print(f"Cache lookup 2: {'HIT' if cached2 else 'MISS'}")
console.print(f"Cache stats: {cache.cache_stats()}")
console.print(f"Session memory (user_001): {len(cache.get_session_context('user_001'))} entries")

---
## Block 10 — Observability Everywhere
> Trace retrieval path · Chunk rankings & scores · Token attribution · Failure case logging · Dashboards & alerts

In [ ]:
class ObservabilityLayer:
    """
    Block 10: Full observability — retrieval path tracing, score logging,
    token attribution, failure logging, and dashboard-ready export.
    """

    def __init__(self):
        self.traces: List[Dict] = []
        self.failure_log: List[Dict] = []

    def record_trace(
        self,
        query: str,
        hybrid_results: List[Tuple[Document, float]],
        reranked: List[Tuple[Document, float]],
        scored: List[ScoredChunk],
        gate_passed: bool,
        gen_result: Dict,
        citations: List[CitationRecord],
        eval_result: Dict,
        cache_hit: bool = False,
    ) -> Dict:
        trace = {
            "trace_id": hashlib.md5((query + str(time.time())).encode()).hexdigest()[:8],
            "timestamp": datetime.datetime.utcnow().isoformat(),
            "query": query,
            "cache_hit": cache_hit,

            # Retrieval path
            "hybrid_candidates": [
                {"source": d.source, "page": d.page, "rrf_score": round(s, 4)}
                for d, s in hybrid_results
            ],
            "reranked": [
                {"source": d.source, "page": d.page, "ce_score": round(s, 4)}
                for d, s in reranked
            ],
            "scored_chunks": [
                {
                    "source": sc.doc.source, "page": sc.doc.page,
                    "confidence": round(sc.final_confidence, 4),
                    "freshness": round(sc.freshness_score, 4),
                    "trust": round(sc.trust_score, 4),
                    "overlap": round(sc.overlap_score, 4),
                }
                for sc in scored
            ],

            # Gate
            "gate_passed": gate_passed,

            # Generation
            "input_tokens": gen_result.get("input_tokens"),
            "output_tokens": gen_result.get("output_tokens"),
            "answer_length": len(gen_result.get("answer", "")),

            # Citations
            "citations": [
                {"index": c.index, "source": c.source, "page": c.page,
                 "confidence": round(c.confidence, 4)}
                for c in citations
            ],

            # Quality
            "faithfulness": round(eval_result["faithfulness"], 4),
            "hallucination_risk": round(eval_result["hallucination_risk"], 4),
            "retrieval_recall": round(eval_result["retrieval_recall"], 4),
        }
        self.traces.append(trace)

        # Alert on high hallucination risk
        if eval_result["hallucination_risk"] > 0.20:
            self._alert(trace, f"High hallucination risk: {eval_result['hallucination_risk']:.2%}")

        return trace

    def log_failure(self, query: str, reason: str) -> None:
        entry = {"query": query, "reason": reason,
                 "timestamp": datetime.datetime.utcnow().isoformat()}
        self.failure_log.append(entry)
        logger.warning(f"[FAILURE] {reason} | query='{query[:60]}'")

    def _alert(self, trace: Dict, message: str) -> None:
        logger.warning(f"[ALERT] trace_id={trace['trace_id']} — {message}")

    def dashboard(self) -> None:
        """Print a simple observability dashboard."""
        if not self.traces:
            console.print("No traces recorded yet.")
            return
        df = pd.DataFrame(self.traces)
        console.print(Panel(
            f"Total traces:          {len(df)}\n"
            f"Cache hit rate:        {df['cache_hit'].mean():.1%}\n"
            f"Gate pass rate:        {df['gate_passed'].mean():.1%}\n"
            f"Avg faithfulness:      {df['faithfulness'].mean():.2%}\n"
            f"Avg hallucin. risk:    {df['hallucination_risk'].mean():.2%}\n"
            f"Avg retrieval recall:  {df['retrieval_recall'].mean():.2%}\n"
            f"Avg input tokens:      {df['input_tokens'].mean():.0f}\n"
            f"Avg output tokens:     {df['output_tokens'].mean():.0f}\n"
            f"Failures logged:       {len(self.failure_log)}",
            title="[10] Observability Dashboard",
            border_style="magenta"
        ))
        return df

    def export_traces(self, path: str = "/tmp/rag_traces.json") -> None:
        with open(path, "w") as f:
            json.dump(self.traces, f, indent=2)
        logger.info(f"Traces exported to {path}")


obs = ObservabilityLayer()
trace = obs.record_trace(
    query=QUERY,
    hybrid_results=hybrid_results,
    reranked=reranked_results,
    scored=scored_chunks,
    gate_passed=gate_passed,
    gen_result=gen_result,
    citations=citations,
    eval_result=eval_result,
    cache_hit=False,
)

obs.dashboard()
obs.export_traces()

console.print(f"\n[bold]Trace ID:[/bold] {trace['trace_id']}")
console.print(json.dumps(trace, indent=2))

---
## End-to-End Pipeline — Single Function

Wires all 10 blocks together into one callable `rag_query()` function.

In [ ]:
def rag_query(
    query: str,
    user_id: str = "anonymous",
    top_k_hybrid: int = 8,
    top_n_rerank: int = 4,
    confidence_threshold: float = 0.30,
    verbose: bool = True,
) -> Dict:
    """
    Full RAG pipeline with zero hallucination.
    Returns a dict with answer, citations, trace, and quality metrics.
    """
    if verbose:
        console.rule(f"[bold blue]RAG Query[/bold blue]")
        console.print(f"[bold]Q:[/bold] {query}")

    # ── Block 9: Check cache first ─────────────────────────────────────────────
    cached = cache.get(query)
    if cached:
        if verbose:
            console.print("[yellow]⚡ Cache HIT — returning cached response[/yellow]")
        return {**cached, "cache_hit": True}

    # ── Block 2: Hybrid retrieval ──────────────────────────────────────────────
    hy_results = retriever.retrieve(query, top_k=top_k_hybrid)

    # ── Block 3: Reranking ────────────────────────────────────────────────────
    rr_results = reranker.rerank(query, hy_results, top_n=top_n_rerank)

    # ── Block 4: Confidence scoring ───────────────────────────────────────────
    sc_chunks = conf_scorer.score(query, rr_results)

    # ── Block 7: Confidence gate ──────────────────────────────────────────────
    g_passed, g_reason = gate.check(sc_chunks)

    if not g_passed:
        obs.log_failure(query, g_reason)
        result = {
            "answer": "I cannot answer this question — insufficient evidence found in the documents.",
            "citations": [],
            "gate_passed": False,
            "gate_reason": g_reason,
            "cache_hit": False,
        }
        if verbose:
            console.print(Panel(
                f"[bold red]{result['answer']}[/bold red]\n\nReason: {g_reason}",
                title="Insufficient Evidence", border_style="red"
            ))
        return result

    # ── Block 5: Constrained generation ──────────────────────────────────────
    g_result = generator.generate(query, sc_chunks)

    # ── Block 6: Citation building ────────────────────────────────────────────
    cits = citation_builder.build(sc_chunks)
    footer = citation_builder.format_citation_footer(cits)
    final_answer = g_result["answer"] + "\n" + footer

    # ── Block 8: Eval ─────────────────────────────────────────────────────────
    ev = eval_suite.run_case(
        query=query, answer=g_result["answer"],
        chunks=sc_chunks, known_relevant=[], gate_passed=g_passed
    )

    # ── Block 10: Trace ───────────────────────────────────────────────────────
    tr = obs.record_trace(
        query=query, hybrid_results=hy_results, reranked=rr_results,
        scored=sc_chunks, gate_passed=g_passed, gen_result=g_result,
        citations=cits, eval_result=ev, cache_hit=False,
    )

    # ── Block 9: Store in cache + session ─────────────────────────────────────
    result = {
        "answer": final_answer,
        "citations": cits,
        "gate_passed": True,
        "faithfulness": ev["faithfulness"],
        "hallucination_risk": ev["hallucination_risk"],
        "trace_id": tr["trace_id"],
        "input_tokens": g_result.get("input_tokens"),
        "output_tokens": g_result.get("output_tokens"),
        "cache_hit": False,
    }
    cache.set(query, result)
    cache.add_to_session(user_id, query, g_result["answer"])

    if verbose:
        console.print(Panel(final_answer, title="Final Response", border_style="green"))
        console.print(
            f"Faithfulness: [cyan]{ev['faithfulness']:.2%}[/cyan] | "
            f"Hallucination risk: [cyan]{ev['hallucination_risk']:.2%}[/cyan] | "
            f"Trace: [yellow]{tr['trace_id']}[/yellow]"
        )

    return result


# ── Demo queries ──────────────────────────────────────────────────────────────
print("\n" + "="*70)
r1 = rag_query("What is retrieval-augmented generation and why does it reduce hallucination?")

print("\n" + "="*70)
r2 = rag_query("Who invented the telephone?")  # out-of-corpus → should be blocked

print("\n" + "="*70)
# Second call — same query as r1 — should be a cache hit
r3 = rag_query("What is retrieval-augmented generation and why does it reduce hallucination?")
print(f"Cache hit on r3: {r3['cache_hit']}")

In [ ]:
# Final observability dashboard across all queries
obs.dashboard()
obs.export_traces("/tmp/rag_traces_final.json")
console.print("\n[bold green]Pipeline complete. All traces exported.[/bold green]")